In [ ]:
import pandas as pd
import glob

# ---------------------------------------
# STEP 1: Detect Excel files
# ---------------------------------------
files = glob.glob("/content/*.xlsx")
print("Files detected:", len(files))

required_columns = [
    "PM2.5", "PM10", "NO", "NO2", "NOx", "NH3",
    "SO2", "CO", "Ozone", "Benzene", "Toluene",
    "RH", "WS", "WD", "SR", "BP", "VWS",
    "TOT-RF", "RF", "AT"
]

final_data = []

# ---------------------------------------
# STEP 2: Process Each File
# ---------------------------------------
for file in files:

    # Read raw sheet
    raw = pd.read_excel(file, header=None)

    # Extract city name
    city_row = raw[raw.iloc[:,0] == "City"]
    city_name = city_row.iloc[0,1]

    # Find header row
    header_index = raw[raw.iloc[:,0] == "From Date"].index[0]

    # Read actual data table
    df = pd.read_excel(file, header=header_index)

    # Remove rows where Date is empty
    df = df[df["From Date"].notna()]

    # Keep only required columns
    existing_cols = [col for col in required_columns if col in df.columns]
    df = df[["From Date"] + existing_cols]

    # Rename Date column
    df.rename(columns={"From Date": "Date"}, inplace=True)

    # Convert Date properly
    df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

    # Remove invalid date rows
    df = df[df["Date"].notna()]

    # Add missing columns as NA (for consistency across cities)
    for col in required_columns:
        if col not in df.columns:
            df[col] = pd.NA

    # Reorder columns
    df = df[["Date"] + required_columns]

    # Add city column
    df.insert(0, "City", city_name)

    final_data.append(df)

# ---------------------------------------
# STEP 3: Merge All Files
# ---------------------------------------
final_df = pd.concat(final_data, ignore_index=True)

# Clean string "None"
final_df.replace("None", pd.NA, inplace=True)

# ---------------------------------------
# STEP 4: Save
# ---------------------------------------
final_df.to_csv("Merged_Air_Quality_Data.csv", index=False)

print("✅ Merging Completed Successfully!")
print("Final Shape:", final_df.shape)
print("\nRows per City:\n")
print(final_df["City"].value_counts())

Files detected: 8
✅ Merging Completed Successfully!
Final Shape: (140230, 22)

Rows per City:

City
Anantapur            17542
Kadapa               17542
Amaravati            17541
Rajamahendravaram    17521
Tirupati             17521
Visakhapatnam        17521
Vijayawada           17521
Chittoor             17521
Name: count, dtype: int64


In [ ]:
import pandas as pd
import glob
import os
import re

files = glob.glob("/content/*.csv")

print("Files detected:", len(files))

final_columns = [
    "City", "Date",
    "PM2.5", "PM10", "NO", "NO2", "NOx", "NH3",
    "SO2", "CO", "Ozone", "Benzene", "Toluene",
    "RH", "WS", "WD", "SR", "BP", "VWS",
    "TOT-RF", "RF", "AT"
]

merged_data = []

for file in files:

    # -------- Extract Clean City Name --------
    filename = os.path.basename(file).split(".")[0]
    parts = filename.split("_")
    city_name = parts[-3]   # Vijayawada, Tirupati etc.

    print("Processing:", city_name)

    # -------- Read CSV --------
    df = pd.read_csv(file, encoding='latin1')  # fixes Âµ issue

    # -------- Clean Column Names --------
    df.columns = df.columns.str.strip()

    # Remove units like (µg/m³), (mm), etc.
    df.columns = df.columns.str.replace(r"\(.*?\)", "", regex=True)

    # Remove extra spaces
    df.columns = df.columns.str.replace(" ", "")

    # Rename Timestamp to Date
    if "Timestamp" in df.columns:
        df.rename(columns={"Timestamp": "Date"}, inplace=True)

    # Add City column
    df["City"] = city_name

    merged_data.append(df)

# -------- Merge All Files --------
final_df = pd.concat(merged_data, ignore_index=True)

# -------- Keep Only Required Columns --------
final_df = final_df[final_columns]

# -------- Convert Date --------
final_df["Date"] = pd.to_datetime(final_df["Date"], errors='coerce')

# -------- Save --------
final_df.to_csv("Merged_Air_Quality_Data_2023.csv", index=False)

print("✅ Merging Completed Successfully")
print("Final shape:", final_df.shape)

Files detected: 8
Processing: Rajamahendravaram
Processing: Tirupati
Processing: Anantapur
Processing: Kadapa
Processing: Chittoor
Processing: Vijayawada
Processing: Visakhapatnam
Processing: Amaravati
✅ Merging Completed Successfully
Final shape: (70080, 22)


In [1]:
import pandas as pd

# --------------------------------------------------
# STEP 1: Load Both Files
# --------------------------------------------------
df1 = pd.read_csv("Merged_Air_Quality_Data.csv")
df2 = pd.read_csv("Merged_Air_Quality_Data_2023.csv")

# Convert Date column
df1["Date"] = pd.to_datetime(df1["Date"])
df2["Date"] = pd.to_datetime(df2["Date"])

# --------------------------------------------------
# STEP 2: Merge Files
# --------------------------------------------------
df = pd.concat([df1, df2], ignore_index=True)

print("After merge:", df.shape)

# --------------------------------------------------
# STEP 3: Create Full Hourly Date Range (2023–2025)
# --------------------------------------------------
full_date_range = pd.date_range(
    start="2023-01-01 00:00",
    end="2025-12-31 23:00",
    freq="H"
)

# --------------------------------------------------
# STEP 4: Ensure Every City Has Full Hourly Data
# --------------------------------------------------
cities = df["City"].unique()

complete_data = []

for city in cities:

    city_df = df[df["City"] == city]

    # Create full date index
    city_df = city_df.set_index("Date")
    city_df = city_df.reindex(full_date_range)

    city_df["City"] = city

    city_df = city_df.reset_index()
    city_df.rename(columns={"index": "Date"}, inplace=True)

    complete_data.append(city_df)

final_df = pd.concat(complete_data, ignore_index=True)

# --------------------------------------------------
# STEP 5: Sort Properly (City-wise + Date-wise)
# --------------------------------------------------
final_df = final_df.sort_values(["City", "Date"])

# --------------------------------------------------
# STEP 6: Save Final File
# --------------------------------------------------
final_df.to_csv("Final_Merged_2023_2025_Hourly.csv", index=False)

print("✅ Final Shape:", final_df.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'Merged_Air_Quality_Data.csv'